# 06 — RoBERTa (AutoModelForMultipleChoice) for the Smart MCQ Solver

## 1. Setup

Imports + reproducibility seeds + device detection.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup

In [2]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


## 2. Configuration

Key choices i made here:
- **`max_len=128`** — per-option length (prompt + ONE option), not all 5 at once
- **`batch_size=8`** — each example is 5x wider than DeBERTa's per-option batches (5 choices stacked)
- **`lr=2e-5`** — standard transformer fine-tuning LR
- **`epochs=5`** — transformers converge fast on small datasets
- **`warmup_ratio=0.1`** — 10% of steps ramp LR from 0 → max before decaying

In [4]:
CFG = dict(
    model_name   = 'roberta-base',
    max_len      = 128,     # per-option length; prompt + one option, not all 5 at once
    batch_size   = 8,       # 5x wider per example than DeBERTa's per-option batches (5 choices stacked)
    lr           = 2e-5,
    weight_decay = 0.01,
    epochs       = 5,
    warmup_ratio = 0.1,
    patience     = 3,
    seed         = SEED,
)

In [5]:
DATA_DIR = Path('/kaggle/input/competitions/smart-mcq-solver-challenge')
if not DATA_DIR.exists():
    DATA_DIR = Path('data')

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
MODEL_DIR  = OUTPUT_DIR / 'models'
PRED_DIR   = OUTPUT_DIR / 'predictions'
for d in (OUTPUT_DIR, MODEL_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [6]:
# Answer <-> int maps
ANSWER_MAP  = {'A':0,'B':1,'C':2,'D':3,'E':4}
REVERSE_MAP = {v:k for k,v in ANSWER_MAP.items()}
OPTION_COLS = list('ABCDE')
print(CFG)

{'model_name': 'roberta-base', 'max_len': 128, 'batch_size': 8, 'lr': 2e-05, 'weight_decay': 0.01, 'epochs': 5, 'warmup_ratio': 0.1, 'patience': 3, 'seed': 42}


## 3. Load data + stratified split

In [7]:
def load_raw():
    """Load competition CSVs."""
    train_df = pd.read_csv(DATA_DIR / 'train.csv')
    test_df  = pd.read_csv(DATA_DIR / 'test.csv')
    return train_df, test_df

In [8]:
def lowercase_columns(df, cols=['prompt']+OPTION_COLS):
    """Lowercase + strip whitespace on every text column."""
    df = df.copy()
    for col in cols:
        df[col] = df[col].astype(str).str.lower().str.strip()
    return df

In [9]:
def stratified_split(train_df, val_size=0.2, seed=SEED):
    """Manual stratified split on the 'answer' column."""
    np.random.seed(seed)
    train_idx, val_idx = [], []
    for ans in 'ABCDE':
        idx = train_df[train_df['answer']==ans].index.tolist()
        np.random.shuffle(idx)
        cut = int(len(idx)*(1.0-val_size))
        train_idx += idx[:cut]
        val_idx   += idx[cut:]
    tr = train_df.loc[train_idx].reset_index(drop=True)
    va = train_df.loc[val_idx].reset_index(drop=True)
    return tr, va

In [10]:
# Load + clean
train_raw, test_df = load_raw()
train_raw = lowercase_columns(train_raw)
test_df   = lowercase_columns(test_df)

# Stratified 80/20
tr_df, va_df = stratified_split(train_raw)
print(f'train: {len(tr_df)} | val: {len(va_df)} | test: {len(test_df)}')

train: 1599 | val: 401 | test: 500
